# Практика · RAG і векторні бази

> Теорія: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

Зошит самодостатній: усе, що тут відбувається, пояснюється на місці, лекцію
відкривати не обовʼязково.

**Задача, яку ми розвʼязуємо.** Побудувати пошук по описах програм, установлених
на цій машині, і **чесно його оцінити** — тобто не «здається, працює», а числом,
на еталоні, який ми не самі вигадали. Потім зібрати з цього пошуку конструкцію
RAG і подивитись, де в ній вузьке місце.

Що зробимо:

1. дістанемо справжні пари «запит → потрібний документ» із бази пакетів;
2. побудуємо **інвертований індекс** і заміряємо, скільки роботи він економить;
3. напишемо **BM25** з нуля;
4. напишемо **recall@k** і **MRR**, звіримо свою TF-IDF з `scikit-learn`;
5. зробимо **щільні вектори** стисненням тієї самої матриці й порівняємо;
6. **навчимо двобічний енкодер** із негативами з пачки;
7. заміряємо **наближений пошук** `faiss`: повнота проти швидкості;
8. зберемо підказку для RAG і порахуємо **стелю** всієї конструкції.

> ⏱ Зошит навчає один невеликий енкодер і будує кілька індексів.
> Заміряно: **близько двох хвилин процесорного часу** на чотирьох ядрах без
> відеокарти. За настінним годинником на завантаженій машині вийде помітно
> більше — саме тому зошит скрізь міряє процесорний час, а не годинник.
> Найдовші кроки — навчання енкодера й побудова індексів `faiss`.

⚠️ **Числа в тебе будуть інші, і це нормально.** Дані беруться з **твоєї**
системи: скільки пакетів установлено, такий і корпус. Форма висновків
відтворюється, конкретні числа — ні. Тому зошит друкує **свої** числа, а не
звіряється з чужими.

In [ ]:
# Фіксуємо кількість потоків ДО імпорту numpy: інакше бібліотеки лінійної алгебри
# розповзаються по всіх ядрах, і будь-який замір часу перестає щось означати.
import os
for var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[var] = '1'

import re, math, json, time, random, subprocess, collections, statistics
from zlib import crc32          # стабільний хеш; чому не вбудований hash() — нижче
import numpy as np

# Один спільний генератор випадковості на весь зошит: усе відтворюється.
SEED = 0
rng = np.random.default_rng(SEED)

print('numpy', np.__version__)
print('потоки зафіксовано:', os.environ['OMP_NUM_THREADS'])
started_at = time.process_time()

## 1 · Звідки беремо дані

Курс не тримає даних у репозиторії. Корпус ми беремо **з твоєї машини**: це база
встановлених пакетів. У кожного пакета є два поля, які написала жива людина:

* `%{SUMMARY}` — один рядок, що це за програма;
* `%{DESCRIPTION}` — кілька речень докладніше.

Це готовий еталон пошуку, якого ніхто не розмічав спеціально: **резюме поводиться
як запит, опис — як документ, і ми точно знаємо, який документ правильний**.

Читаємо базу командою `rpm`. Якщо в тебе дистрибутив на основі Debian, пакетів
`rpm` немає, і те саме дає `dpkg-query`; відповідну команду зошит надрукує, але
**чесно попереджаємо: цей шлях ми не перевіряли**, бо машини з Debian у нас під
рукою не було.

In [ ]:
FIELDS = ('NAME', 'SUMMARY', 'DESCRIPTION')

def load_packages(min_words=20):
    """Читає базу пакетів. Повертає список словників.

    Роздільники беремо службові (\x1f між полями, \x1e між записами):
    в описах трапляються і коми, і переноси рядків, тому звичайні роздільники
    не годяться."""
    query_format = '\x1f'.join('%{' + f + '}' for f in FIELDS) + '\x1e'
    try:
        raw = subprocess.run(['rpm', '-qa', '--qf', query_format],
                             capture_output=True, text=True, timeout=180).stdout
    except FileNotFoundError:
        raise SystemExit(
            'Команди rpm на цій машині немає.\n'
            'На дистрибутивах з Debian те саме дає:\n'
            "    dpkg-query -W -f '${Package}\\x1f${binary:Summary}\\x1f${Description}\\x1e'\n"
            'Цей шлях ми НЕ перевіряли — якщо працює, підстав його сюди замість rpm.')
    records = []
    for chunk in raw.split('\x1e'):
        if not chunk.strip():
            continue
        parts = chunk.split('\x1f')
        if len(parts) != len(FIELDS):
            continue
        rec = dict(zip((f.lower() for f in FIELDS), (p.strip() for p in parts)))
        # беремо лише пакети, у яких є і опис пристойної довжини, і резюме
        if (rec['description'] and rec['description'] != '(none)'
                and len(rec['description'].split()) >= min_words
                and rec['summary'] and rec['summary'] != '(none)'):
            records.append(rec)
    records.sort(key=lambda r: r['name'])     # порядок фіксований — щоб усе відтворювалось
    return records

packages = load_packages()
if len(packages) < 200:
    raise SystemExit(f'Знайшлось лише {len(packages)} придатних пакетів. '
                     'Для цього зошита треба щонайменше 200 — інакше метрики '
                     'будуть шумом. Спробуй машину, де встановлено більше програм.')
print(f'придатних пакетів: {len(packages)}')
print('перший запис:', packages[0]['name'], '·', packages[0]['summary'])

### Перш ніж рахувати — подивись на дані очима

Це найдешевша перевірка в усьому зошиті, і вона регулярно рятує від
багатоденної помилки. Подивимось на одну пару.

In [ ]:
example = packages[len(packages) // 3]
print('НАЗВА  :', example['name'])
print('ЗАПИТ  :', example['summary'])
print('ДОКУМЕНТ:')
print(example['description'][:400].replace('\n', ' '))

У базі є пастка, яку видно тільки якщо пошукати навмисне: для програм, що
мають і 64-бітну, і 32-бітну збірку, зберігаються **два окремі записи з
однаковим резюме й однаковим описом**.

Для еталона пошуку це поломка. Виходить, що в запиту насправді **два** однаково
правильні документи, а зараховується лише один — і система, яка поставила
першим «не того» близнюка, дістає штраф ні за що. Знайдемо такі записи й
приберемо.

In [ ]:
seen_pairs = set()
unique_packages = []
duplicates = []
for record in packages:
    key = (record['summary'], record['description'])
    if key in seen_pairs:
        duplicates.append(record['name'])
        continue
    seen_pairs.add(key)
    unique_packages.append(record)

print(f'було {len(packages)} · стало {len(unique_packages)} · '
      f'прибрано {len(duplicates)} ({len(duplicates)/len(packages):.1%})')
print('приклади прибраних:', ', '.join(sorted(set(duplicates))[:6]))
packages = unique_packages

## 2 · Токенізація й розмір задачі

Розрізаємо тексти на слова. Опис пакета — англійський технічний текст, тож берем
латинські послідовності літер і опускаємо регістр. Апостроф лишаємо всередині
слова, щоб `don't` не розпалось надвоє.

In [ ]:
WORD = re.compile(r"[a-z]+(?:['’][a-z]+)*")

def tokens(text):
    """Текст -> список слів у нижньому регістрі."""
    return WORD.findall(text.lower())

documents = [tokens(r['description']) for r in packages]
queries = [tokens(r['summary']) for r in packages]
N = len(documents)

doc_len = [len(d) for d in documents]
query_len = [len(q) for q in queries]
print(f'документів (і запитів) : {N}')
print(f'довжина документа      : медіана {statistics.median(doc_len)}, '
      f'середня {sum(doc_len)/N:.1f}')
print(f'довжина запиту         : медіана {statistics.median(query_len)}, '
      f'середня {sum(query_len)/N:.1f}')

### Наскільки задача взагалі розвʼязна лексично

Перед тим як щось будувати, порахуємо просту річ: **скільки слів запиту взагалі
присутні в потрібному документі**. Це не метрика системи — це властивість
самого стенду, і вона одразу каже, чого від нього чекати.

In [ ]:
shared = []
for i in range(N):
    q, d = set(queries[i]), set(documents[i])
    shared.append(len(q & d) / max(1, len(q)))

no_common = sum(1 for s in shared if s == 0)
all_common = sum(1 for s in shared if s == 1)
print(f'середня частка слів запиту, що є в документі : {np.mean(shared):.4f}')
print(f'запитів, де спільних слів немає зовсім       : {no_common} '
      f'({no_common/N:.2%})')
print(f'запитів, де в документі є ВСІ слова запиту   : {all_common} '
      f'({all_common/N:.2%})')
print()
print('Читай це так: стенд прихильний до лексичного пошуку за побудовою —')
print('резюме й опис писала одна людина, і вона вживала ті самі слова.')

## 3 · Інвертований індекс

Наївний пошук порівнює запит з усіма документами. Інвертований індекс замінює
таблицю «документ → його слова» на протилежну: **слово → список документів, у
яких воно є**. Тоді запит чіпає лише свої слова.

Заміряємо, скільки роботи це економить саме на наших даних.

In [ ]:
postings = collections.defaultdict(list)     # слово -> список номерів документів
for doc_id, doc in enumerate(documents):
    # sorted() тут не для краси: порядок обходу множини рядків у Python
    # змінюється від запуску до запуску, і без сортування словник щоразу
    # виходив би в іншому порядку — а з ним і всі похідні числа
    for word in sorted(set(doc)):
        postings[word].append(doc_id)

document_frequency = {w: len(ids) for w, ids in postings.items()}
vocabulary = sorted(postings)
V = len(vocabulary)
print(f'різних слів у колекції: {V}')

# скільки документів доводиться торкнутись на середній запит
touched = []
for q in queries:
    ids = set()
    for w in q:
        ids.update(postings.get(w, ()))
    touched.append(len(ids))
print(f'документів на запит: медіана {statistics.median(touched)} з {N}')
print(f'тобто індекс прибирає {1 - statistics.median(touched)/N:.1%} роботи '
      f'на медіанному запиті')

Число, яке щойно надрукувалось, може розчарувати: індекс прибирає далеко не
все. Причина проста й повчальна — **у запитах є службові слова**. Резюме на
кшталт «The GL Vendor-Neutral Dispatch library» містить `the`, а список
входжень слова `the` — це майже вся колекція. Один такий токен зводить нанівець
економію від трьох рідкісних.

Саме тому справжні пошуковики або викидають зі запиту найчастіші слова, або
обробляють їхні списки окремо. Перевіримо цю здогадку числом: викинемо з запитів
двадцять найчастіших слів колекції й подивимось, що буде.

In [ ]:
# ключ сортування — пара (частота, слово): без другої складової слова з
# однаковою частотою впорядковувались би довільно, і список щоразу був би інший
stopwords = {word for word, _ in
             sorted(document_frequency.items(), key=lambda kv: (-kv[1], kv[0]))[:20]}

touched_without_stopwords = []
for q in queries:
    ids = set()
    for w in q:
        if w in stopwords:
            continue
        ids.update(postings.get(w, ()))
    touched_without_stopwords.append(len(ids))

print('двадцять найчастіших слів:', ' '.join(sorted(stopwords)))
print(f'документів на запит було  : {statistics.median(touched)}')
print(f'документів на запит стало : {statistics.median(touched_without_stopwords)}')
print(f'тепер індекс прибирає '
      f'{1 - statistics.median(touched_without_stopwords)/N:.1%} роботи')

Найчастіші й найрідкісніші слова — щоб побачити, з чим ми маємо справу. Саме
через цю нерівномірність рідкісні слова водночас швидкі й корисні.

In [ ]:
by_frequency = sorted(document_frequency.items(), key=lambda kv: -kv[1])
print('найчастіші слова (слово, у скількох документах):')
for word, count in by_frequency[:8]:
    print(f'  {word:<14} {count}')
rare = sum(1 for c in document_frequency.values() if c == 1)
print(f'\nслів, що трапились рівно в одному документі: {rare} '
      f'({rare/V:.1%} словника)')

## 4 · BM25 з нуля

BM25 складається з трьох ідей, і кожна — окремий множник.

1. **idf** — рідкісне слово важить більше:
   `idf(w) = ln(1 + (N − df(w) + 0.5) / (df(w) + 0.5))`
2. **насичення частоти** — десять згадок не вдесятеро важливіші за одну;
   швидкість насичення задає `k1`.
3. **нормалізація довжини** — довгий документ не має вигравати тільки тому, що
   він довгий; силу штрафу задає `b`.

Разом:

```
BM25(q, d) = Σ idf(w) · f(w,d)·(k1+1) / ( f(w,d) + k1·(1 − b + b·dl/avgdl) )
```

Пишемо це так, щоб рахувати через інвертований індекс, а не перебором.

In [ ]:
average_length = sum(doc_len) / N

def idf(word):
    df = document_frequency.get(word, 0)
    return math.log(1 + (N - df + 0.5) / (df + 0.5))

def build_bm25_index(k1=1.5, b=0.75):
    """слово -> [(номер документа, готовий внесок)].

    Внесок документа за слово не залежить від запиту, тому його рахують один раз
    наперед — і під час запиту лишається тільки додавання."""
    index = collections.defaultdict(list)
    for doc_id, doc in enumerate(documents):
        counts = collections.Counter(doc)
        length_factor = 1 - b + b * len(doc) / average_length
        for word, freq in counts.items():
            weight = idf(word) * freq * (k1 + 1) / (freq + k1 * length_factor)
            index[word].append((doc_id, weight))
    return index

t0 = time.process_time()
bm25_index = build_bm25_index()
print(f'індекс BM25 побудовано за {time.process_time() - t0:.2f} с процесорних')

def bm25_scores(query_words):
    """Оцінки всіх документів для одного запиту."""
    scores = np.zeros(N, dtype=np.float32)
    for word in query_words:
        for doc_id, weight in bm25_index.get(word, ()):
            scores[doc_id] += weight
    return scores

Перевіримо руками на одному запиті: що знайде BM25 і чи буде правильний
документ нагорі.

In [ ]:
probe = len(packages) // 3          # той самий пакет, що ми дивились очима вище
scores = bm25_scores(queries[probe])
top = np.argsort(-scores)[:5]

print('ЗАПИТ:', packages[probe]['summary'])
print('ПОТРІБНИЙ ДОКУМЕНТ:', packages[probe]['name'])
print('\nвидача BM25:')
for place, doc_id in enumerate(top, 1):
    mark = '  <-- правильний' if doc_id == probe else ''
    print(f'  {place}. {scores[doc_id]:7.3f}  {packages[doc_id]["name"]}{mark}')

## 5 · Чим міряти: recall@k і MRR

Пошук видає впорядкований список, тож і міряти треба положення потрібного
документа в ньому.

* **recall@k** — частка запитів, у яких потрібний документ потрапив у перші `k`.
* **MRR** — середній обернений ранг: перше місце дає 1, друге 0.5, третє 1/3.
  Позиція важить, і перші місця важать непропорційно більше.

Пишемо обидві та проганяємо всі запити.

In [ ]:
def ranks_of_gold(score_function):
    """Для кожного запиту — позиція правильного документа у видачі (з 1)."""
    out = np.zeros(N, dtype=np.int32)
    for i in range(N):
        scores = score_function(i)
        # скільки документів дістали оцінку СТРОГО більшу за правильний —
        # це і є кількість тих, хто стоїть вище
        better = int((scores > scores[i]).sum())
        out[i] = better + 1
    return out

def report(ranks):
    result = {f'recall@{k}': float((ranks <= k).mean()) for k in (1, 5, 10)}
    result['MRR'] = float((1.0 / ranks).mean())
    return result

t0 = time.process_time()
bm25_ranks = ranks_of_gold(lambda i: bm25_scores(queries[i]))
bm25_report = report(bm25_ranks)
print(f'пораховано за {time.process_time() - t0:.1f} с процесорних')
for key, value in bm25_report.items():
    print(f'  BM25 {key:<10} {value:.4f}')

### Обережно з нічиїми

У формулі вище ранг рахується як «скільки документів мають оцінку **строго
більшу**». Це важливо: якщо запит не має жодного спільного слова з колекцією,
усі оцінки дорівнюють нулю, і за таким підрахунком правильний документ дістає
ранг 1 — тобто ми б самі собі приписали успіх.

Перевіримо, чи є в нас такі запити, і порахуємо чесний варіант, у якому нічия
розвʼязується не на нашу користь.

In [ ]:
def ranks_pessimistic(score_function):
    """Нічия трактується проти нас: усі, хто має таку саму оцінку, стоять вище."""
    out = np.zeros(N, dtype=np.int32)
    for i in range(N):
        scores = score_function(i)
        better = int((scores > scores[i]).sum())
        same = int((scores == scores[i]).sum()) - 1
        out[i] = better + same + 1
    return out

pessimistic = ranks_pessimistic(lambda i: bm25_scores(queries[i]))
diff = int((pessimistic != bm25_ranks).sum())
print(f'запитів, де нічия щось міняє: {diff} з {N}')
print(f'MRR оптимістичний : {(1.0/bm25_ranks).mean():.4f}')
print(f'MRR песимістичний : {(1.0/pessimistic).mean():.4f}')
print()
print('Далі користуємось песимістичним: він не дарує балів за порожню видачу.')
bm25_ranks = pessimistic
bm25_report = report(bm25_ranks)

## 6 · Своя TF-IDF проти бібліотечної

Перш ніж будувати щось складніше, звіримо власний код із бібліотечним. Це
найкорисніша перевірка в зошиті: видно, що всередині `scikit-learn` немає магії.

Косинусна близькість TF-IDF — не BM25, але вона побудована з тих самих деталей,
і її `scikit-learn` уміє. Зробимо свою версію й порівняємо **ранжування**
(саме воно нас цікавить), а не абсолютні числа: різні реалізації по-різному
згладжують idf, і ваги в них не мусять збігатися.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

# бібліотечна версія: свій токенізатор передаємо явно, щоб порівнювати однакове
vectorizer = TfidfVectorizer(analyzer=tokens, norm='l2', sublinear_tf=True)
library_docs = vectorizer.fit_transform(r['description'] for r in packages)
library_queries = vectorizer.transform(r['summary'] for r in packages)
print('матриця документів:', library_docs.shape)

# наша версія: ті самі формули руками
feature_index = {w: i for i, w in enumerate(vectorizer.get_feature_names_out())}
n_docs = library_docs.shape[0]

def our_tfidf(list_of_token_lists):
    rows = np.zeros((len(list_of_token_lists), len(feature_index)), dtype=np.float64)
    for i, toks in enumerate(list_of_token_lists):
        for word, freq in collections.Counter(toks).items():
            j = feature_index.get(word)
            if j is None:
                continue
            # sublinear_tf: 1 + ln(f); idf зі згладжуванням, як у scikit-learn
            df = int((library_docs[:, j] != 0).sum())
            rows[i, j] = (1 + math.log(freq)) * (math.log((1 + n_docs) / (1 + df)) + 1)
    return normalize(rows)

sample = list(range(0, N, max(1, N // 40)))[:40]   # 40 документів досить для звірки
ours = our_tfidf([documents[i] for i in sample])
theirs = library_docs[sample].toarray()
assert np.allclose(ours, theirs, atol=1e-8), 'наша TF-IDF розійшлася з бібліотечною!'
print('✅ збігається: наша TF-IDF дорівнює бібліотечній на', len(sample), 'документах')

Тепер порахуємо якість пошуку косинусом TF-IDF — уже бібліотечною матрицею,
раз ми переконались, що вона та сама.

In [ ]:
tfidf_similarity = (library_queries @ library_docs.T).toarray()

def tfidf_scores(i):
    return tfidf_similarity[i]

tfidf_ranks = ranks_pessimistic(tfidf_scores)
tfidf_report = report(tfidf_ranks)
for key, value in tfidf_report.items():
    print(f'  TF-IDF {key:<10} {value:.4f}')
print(f'\nBM25 був: MRR {bm25_report["MRR"]:.4f}')

## 7 · Щільні вектори стисненням тієї самої матриці

Найдешевший спосіб дістати щільні вектори — стиснути наявну лексичну матрицю
сингулярним розкладом (це і зветься LSA). Отримаємо не 13 тисяч колонок, а,
скажімо, 256 чисел на документ.

Питання, на яке зараз відповімо числом: **чи додає це щось**. Прогонимо кілька
розмірностей і подивимось, куди йде крива.

In [ ]:
from sklearn.decomposition import TruncatedSVD

dims = [32, 64, 128, 256, 512]
dims = [d for d in dims if d < min(N, library_docs.shape[1])]
lsa_results = {}
lsa_ranks = {}

for dim in dims:
    t0 = time.process_time()
    svd = TruncatedSVD(n_components=dim, random_state=SEED, n_iter=5)
    dense_docs = normalize(svd.fit_transform(library_docs))
    dense_queries = normalize(library_queries @ svd.components_.T)
    similarity = dense_queries @ dense_docs.T
    ranks = ranks_pessimistic(lambda i: similarity[i])
    lsa_results[dim] = report(ranks)
    lsa_ranks[dim] = ranks
    print(f'LSA {dim:>4}: MRR {lsa_results[dim]["MRR"]:.4f} · '
          f'recall@10 {lsa_results[dim]["recall@10"]:.4f} · '
          f'{time.process_time() - t0:.1f} с')

print(f'\nвихідна матриця TF-IDF: MRR {tfidf_report["MRR"]:.4f}')

Подивись на колонку MRR згори вниз і порівняй останній рядок із вихідною
матрицею. Стиснення підходить до неї **знизу**: воно нічого не додає, бо додавати
немає звідки — на вході та сама матриця підрахунків. Що більше напрямків
лишаємо, то ближче до початкової відповіді.

Тепер найцікавіше: **де саме** щільний вектор програє. Розріжемо запити за тим,
скільки їхніх слів є в потрібному документі.

In [ ]:
best_dim = max(lsa_results, key=lambda d: lsa_results[d]['MRR'])
overlap = np.array([len(set(queries[i]) & set(documents[i])) for i in range(N)])
buckets = [(0, 0, 'жодного'), (1, 1, 'одне'), (2, 2, 'два'),
           (3, 3, 'три'), (4, 4, 'чотири'), (5, 99, 'пʼять і більше')]

print(f'recall@10 по зрізах (щільний узято найкращий, {best_dim} напрямків)')
print(f'{"спільних слів":<16}{"запитів":>9}{"BM25":>9}{"TF-IDF":>9}{"щільний":>10}')
for low, high, name in buckets:
    which = np.where((overlap >= low) & (overlap <= high))[0]
    if len(which) == 0:
        continue
    row = [float((r[which] <= 10).mean())
           for r in (bm25_ranks, tfidf_ranks, lsa_ranks[best_dim])]
    print(f'{name:<16}{len(which):>9}{row[0]:>9.4f}{row[1]:>9.4f}{row[2]:>10.4f}')

## 8 · Суміш двох способів

Популярна порада — змішати лексичний і щільний пошук. Найпоширеніший спосіб
зветься **RRF**: беруть ранг документа в кожній системі, перетворюють на
`1 / (60 + ранг)` і додають. Перевага в тому, що складати ранги можна навіть
тоді, коли оцінки систем непорівнянні.

Перевіримо на своїх даних, і не одну вагу, а кілька.

In [ ]:
def rank_positions(score_matrix_row_fn):
    """Позиція КОЖНОГО документа у видачі (0 — перший)."""
    out = np.zeros((N, N), dtype=np.int32)
    for i in range(N):
        order = np.argsort(-score_matrix_row_fn(i))
        out[i, order] = np.arange(N)
    return out

bm25_positions = rank_positions(lambda i: bm25_scores(queries[i]))
svd = TruncatedSVD(n_components=best_dim, random_state=SEED, n_iter=5)
dense_docs = normalize(svd.fit_transform(library_docs))
dense_queries = normalize(library_queries @ svd.components_.T)
dense_similarity = dense_queries @ dense_docs.T
dense_positions = rank_positions(lambda i: dense_similarity[i])

print(f'{"вага щільного":<16}{"MRR":>9}{"recall@10":>12}')
for weight in (0.0, 0.1, 0.25, 0.5, 1.0):
    fused = 1.0 / (60 + bm25_positions + 1) + weight / (60 + dense_positions + 1)
    ranks = ranks_pessimistic(lambda i: fused[i])
    row = report(ranks)
    print(f'{weight:<16.2f}{row["MRR"]:>9.4f}{row["recall@10"]:>12.4f}')

Прочитай **обидві** колонки, а не одну. Зазвичай виходить так: MRR від домішки
падає, а recall@10 може трохи піднятись. Це не суперечність — суміш дістає
кілька документів, яких лексичний пошук не бачить, і водночас псує порядок
нагорі.

Скільки саме тих документів — теж можна порахувати, і це варто робити **до**
змішування.

In [ ]:
bm25_hit = bm25_ranks <= 10
dense_hit = lsa_ranks[best_dim] <= 10
print(f'розвʼязує лише BM25    : {int((bm25_hit & ~dense_hit).sum())}')
print(f'розвʼязує лише щільний : {int((~bm25_hit & dense_hit).sum())}')
print(f'розвʼязують обидва     : {int((bm25_hit & dense_hit).sum())}')
print(f'не розвʼязує ніхто     : {int((~bm25_hit & ~dense_hit).sum())}')
print()
print('Суміш має сенс лише тоді, коли другий рядок помітно більший за нуль.')

## 9 · Двобічний енкодер: вчимо вектори на справжніх парах

Стиснення нічого не додало, бо вчилось на тому самому підрахунку слів. Тепер
дамо векторам **зовнішній сигнал**: навчимо їх на парах «резюме → опис».

Конструкція така:

* два енкодери — один для запитів, другий для документів; кожен перетворює
  текст на вектор фіксованої довжини;
* **негативи з пачки**: у пачці з 64 пар для запиту номер `i` правильний
  документ — `i`-й, а решта 63 документи чужі й слугують неправильними
  відповідями. Вони безкоштовні;
* **контрастна втрата**: звичайна перехресна ентропія по рядку матриці
  схожостей, у якій правильна «мітка» — діагональ;
* **температура** `τ`: ділимо схожості перед softmax, бо косинус живе у вузькому
  проміжку і без цього softmax дає майже рівні ймовірності.

Ознаки слова робимо **символьними n-грамами**: назви програм у відкладених
пакетах наш словник ніколи не бачив, і без n-грам їх нічим представити.
Скільки ознак лишати на документ — окрема ручка, і вона важить більше, ніж
здається: якщо обрізати список надто рано, n-грами покриють лише початок
тексту.

⚠️ Хешуємо через `zlib.crc32`, а **не** через вбудований `hash()`. Вбудований
хеш рядків у Python солиться значенням, яке генерується наново на кожен запуск,
тож той самий код із тим самим зерном давав би різні числа щоразу.

⚠️ І та сама хвороба з іншого боку: **порядок обходу множини рядків теж
змінюється між запусками**. Якщо будувати словник, обходячи `set(...)`, номери
слів щоразу вийдуть інші — а з ними й усі похідні числа. Тому в цьому зошиті
перед кожним перебором множини стоїть `sorted()`. Ми на цьому вже спіймались:
без сортування повнота наближеного пошуку в розділі 10 стрибала на кілька
сотих між запусками при зафіксованому зерні.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.set_num_threads(1)

BUCKETS = 2 ** 16          # у скільки кошиків хешуємо ознаки

def stable_hash(text):
    """Детермінований між запусками, на відміну від вбудованого hash()."""
    return crc32(text.encode()) % BUCKETS

# Межа списку ознак — це теж ручка, і не безневинна. Ознаки складаються так:
# спершу хеші цілих слів, потім символьні n-грами. Кожне слово дає близько
# двадцяти n-грам, тож межа в 400 позицій означала б, що n-грамами покрито лише
# перший десяток слів документа — тобто ліки застосовані до чверті тексту.
MAX_FEATURES = 900

def features(text, max_features=MAX_FEATURES):
    """Ознаки тексту: цілі слова + символьні n-грами по 3 і 4 літери."""
    words = tokens(text)[:64]
    out = [stable_hash('w:' + w) for w in words]
    for word in words:
        padded = '<' + word + '>'
        for n in (3, 4):
            out += [stable_hash(f'c{n}:' + padded[i:i + n])
                    for i in range(len(padded) - n + 1)]
    return out[:max_features] or [0]

# перевіряємо, що хеш справді стабільний
assert stable_hash('gzip') == crc32(b'gzip') % BUCKETS
print('ознак у прикладі:', len(features(packages[0]['summary'])),
      'для запиту і', len(features(packages[0]['description'])), 'для документа')

Ділимо пакети на навчальні й відкладені **за пакетом**, а не за рядком: інакше
той самий пакет опинився б і в навчанні, і в перевірці, і результат був би
завищений. Порівнювати щільний енкодер будемо з BM25 **на тій самій відкладеній
колекції** — інакше порівняння нечесне.

In [ ]:
names = sorted({r['name'] for r in packages})
random.Random(SEED).shuffle(names)
train_names = set(names[:int(0.8 * len(names))])

train = [r for r in packages if r['name'] in train_names]
held_out = [r for r in packages if r['name'] not in train_names]
print(f'навчальних пар: {len(train)} · відкладена колекція: {len(held_out)}')

# ознаки рахуємо ОДИН раз на текст, а не на кожному кроці навчання:
# інакше хешування з'їдає більше часу, ніж саме навчання
train_query_feats = [features(r['summary']) for r in train]
train_doc_feats = [features(r['description']) for r in train]
eval_query_feats = [features(r['summary']) for r in held_out]
eval_doc_feats = [features(r['description']) for r in held_out]
print('ознаки пораховано')

In [ ]:
def pad(batch):
    width = max(len(x) for x in batch)
    return torch.tensor([x + [0] * (width - len(x)) for x in batch])

class BagEncoder(nn.Module):
    """Найпростіший енкодер, у якого є що вчити: усереднений мішок ембедингів."""
    def __init__(self, dim):
        super().__init__()
        # sparse=True: градієнт чіпає лише ті рядки таблиці, які трапились у пачці.
        # З щільним градієнтом оптимізатор щокроку оновлював би мільйони чисел.
        self.emb = nn.EmbeddingBag(BUCKETS, dim, mode='mean', sparse=True)

    def forward(self, x):
        return F.normalize(self.emb(x), dim=-1)

def train_dual_encoder(dim=128, steps=1500, batch_size=64, temperature=0.05, lr=2e-3):
    torch.manual_seed(SEED)
    query_encoder, doc_encoder = BagEncoder(dim), BagEncoder(dim)
    optimizer = torch.optim.SparseAdam(
        list(query_encoder.parameters()) + list(doc_encoder.parameters()), lr=lr)
    picker = random.Random(SEED)
    indices = list(range(len(train)))
    for step in range(steps):
        batch = picker.sample(indices, min(batch_size, len(indices)))
        q = query_encoder(pad([train_query_feats[j] for j in batch]))
        d = doc_encoder(pad([train_doc_feats[j] for j in batch]))
        logits = q @ d.T / temperature            # схожості всіх з усіма в пачці
        target = torch.arange(len(batch))         # правильна відповідь — діагональ
        loss = F.cross_entropy(logits, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if (step + 1) % 500 == 0:
            print(f'  крок {step + 1:>5}: втрата {loss.item():.4f}')
    return query_encoder, doc_encoder

t0 = time.process_time()
query_encoder, doc_encoder = train_dual_encoder()
print(f'навчання зайняло {(time.process_time() - t0)/60:.1f} хв процесорних')

Оцінюємо на відкладених пакетах — тих, яких енкодер під час навчання не бачив.
Поруч рахуємо BM25 **на тій самій колекції**: інакше числа непорівнянні.

In [ ]:
query_encoder.eval(); doc_encoder.eval()
with torch.no_grad():
    eval_q = query_encoder(pad(eval_query_feats)).numpy()
    eval_d = doc_encoder(pad(eval_doc_feats)).numpy()
dense_eval = eval_q @ eval_d.T

M = len(held_out)
eval_docs = [tokens(r['description']) for r in held_out]
eval_queries = [tokens(r['summary']) for r in held_out]
eval_df = collections.Counter(w for d in eval_docs for w in set(d))
eval_avg = sum(len(d) for d in eval_docs) / M

def bm25_on_held_out():
    scores = np.zeros((M, M), dtype=np.float32)
    index = collections.defaultdict(list)
    for i, doc in enumerate(eval_docs):
        counts = collections.Counter(doc)
        length_factor = 0.25 + 0.75 * len(doc) / eval_avg
        for word, freq in counts.items():
            df = eval_df[word]
            weight = math.log(1 + (M - df + 0.5) / (df + 0.5))
            index[word].append((i, weight * freq * 2.5 / (freq + 1.5 * length_factor)))
    for qi, q in enumerate(eval_queries):
        for word in q:
            for i, weight in index.get(word, ()):
                scores[qi, i] += weight
    return scores

def report_square(matrix):
    ranks = np.zeros(M, dtype=np.int32)
    for i in range(M):
        better = int((matrix[i] > matrix[i, i]).sum())
        same = int((matrix[i] == matrix[i, i]).sum()) - 1
        ranks[i] = better + same + 1
    return ({f'recall@{k}': float((ranks <= k).mean()) for k in (1, 5, 10)}
            | {'MRR': float((1.0 / ranks).mean())})

bm25_held = report_square(bm25_on_held_out())
dense_held = report_square(dense_eval)
print(f'{"":<24}{"recall@1":>10}{"recall@10":>12}{"MRR":>9}')
for name, row in (('BM25', bm25_held), ('навчений щільний', dense_held)):
    print(f'{name:<24}{row["recall@1"]:>10.4f}{row["recall@10"]:>12.4f}{row["MRR"]:>9.4f}')

Числа в тебе будуть свої, але напрямок зазвичай той самий: **маленький
енкодер, навчений з нуля на кількох сотнях пар, лексичному пошуку програє**.
Це чесна відповідь на питання «чи варто вчити свій енкодер замість BM25» — і це
**не** відповідь на питання «чи виграла б велика модель, попередньо навчена на
сотнях мільйонів пар». Таку модель тут ніхто не пробував.

Перевіримо ще одну річ, яку легко пропустити: **чи не вивчив енкодер просто
частотність**. Контроль — підставити чужий запит. Якщо якість не впаде, значить
модель не читає запит зовсім.

In [ ]:
shuffled = np.random.default_rng(SEED).permutation(M)
control = eval_q[shuffled] @ eval_d.T
control_hits = float((np.argmax(control, axis=1) == np.arange(M)).mean())
real_hits = float((np.argmax(dense_eval, axis=1) == np.arange(M)).mean())
print(f'влучань у перше місце, справжній запит : {real_hits:.4f}')
print(f'влучань у перше місце, чужий запит     : {control_hits:.4f}')
print()
if real_hits > control_hits * 3:
    print('Контроль пройдено: із чужим запитом система розвалюється,')
    print('отже вона таки читає запит, а не вгадує популярні документи.')
else:
    print('Увага: чужий запит працює майже так само — модель запит не читає.')

## 10 · Наближений пошук: чим платиш за швидкість

На 1733 документах наближений пошук безглуздий — точний перебір і так миттєвий.
Щоб побачити компроміс, потрібен масштаб, і він теж є на твоїй машині: це
українські переклади інтерфейсів у файлах `.mo`. Їх десятки тисяч.

⚠️ Це **інший корпус** і потрібен він лише для одного — показати криву «повнота
проти швидкості». Якості пошуку ми тут не міряємо.

In [ ]:
import glob, gettext

def load_messages(min_length=30):
    """Українські переклади рядків інтерфейсу з системних файлів .mo."""
    out = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)._catalog
        except Exception:
            continue
        for source, target in catalog.items():
            if (isinstance(source, str) and isinstance(target, str)
                    and len(target) > min_length and 'Project-Id' not in target):
                out.append(target)
    return out

messages = load_messages()
print(f'рядків знайдено: {len(messages)}')
if len(messages) < 5000:
    print('Замало для демонстрації масштабу — розділ 10 буде малопоказовим.')
    print('Українські переклади ставляться разом із мовними пакетами системи.')

In [ ]:
UA_WORD = re.compile(r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*")
message_tokens = [UA_WORD.findall(t.lower()) for t in messages]
ua_df = collections.Counter()
for t in message_tokens:
    ua_df.update(sorted(set(t)))
# ЗНОВУ sorted(), і цього разу ціна помилки висока: без нього номери рядків
# у таблиці проєкції залежали б від порядку обходу множин, вектори виходили б
# щоразу інші, і повнота наближеного пошуку стрибала б на кілька сотих між
# запусками — при тому, що зерно генератора зафіксоване
ua_vocab = {w: i for i, w in enumerate(sorted(w for w, c in ua_df.items() if c >= 3))}
print(f'словник (слова, що трапились щонайменше тричі): {len(ua_vocab)}')

# Робимо щільні вектори дешево: випадкова проєкція TF-IDF у 128 вимірів.
# Тут важлива не якість векторів, а те, як поводиться індекс на їхній кількості.
DIM = 128
n_msg = len(messages)
projection = np.zeros((len(ua_vocab), DIM), dtype=np.float32)
for i in range(len(ua_vocab)):
    columns = rng.choice(DIM, 3, replace=False)
    projection[i, columns] = rng.choice([-1.0, 1.0], 3)

vectors = np.zeros((n_msg, DIM), dtype=np.float32)
for i, toks in enumerate(message_tokens):
    counts = collections.Counter(w for w in toks if w in ua_vocab)
    for word, freq in counts.items():
        vectors[i] += ((1 + math.log(freq)) * math.log(n_msg / ua_df[word])
                       * projection[ua_vocab[word]])
vectors /= (np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9)
print(f'вектори: {vectors.shape}, памʼять {vectors.nbytes / 2**20:.1f} МіБ')

In [ ]:
import faiss
faiss.omp_set_num_threads(1)          # без цього час залежить від сусідніх програм

n_queries = min(500, n_msg // 10)
probe_ids = rng.choice(n_msg, n_queries, replace=False)
probe_vectors = vectors[probe_ids].copy()

def timed(fn, repeats=3):
    """Медіана процесорного часу з кількох прогонів."""
    samples = []
    for _ in range(repeats):
        t0 = time.process_time()
        fn()
        samples.append(time.process_time() - t0)
    return statistics.median(samples)

flat = faiss.IndexFlatIP(DIM)
flat.add(vectors)
_, exact_ids = flat.search(probe_vectors, 10)
flat_time = timed(lambda: flat.search(probe_vectors, 10))
print(f'точний перебір: {flat_time / n_queries * 1000:.4f} мс на запит '
      f'({n_msg} документів)')
print('Зверни увагу, наскільки це швидко: перебір — це множення матриць,')
print('а множення матриць процесор робить найкраще з усього.')

In [ ]:
print(f'{"nlist":>7}{"nprobe":>8}{"повнота@10":>13}{"мс/запит":>11}{"швидше":>10}')
ann_rows = []
for nlist in (256, 1024):
    quantizer = faiss.IndexFlatIP(DIM)
    ivf = faiss.IndexIVFFlat(quantizer, DIM, nlist, faiss.METRIC_INNER_PRODUCT)
    ivf.train(vectors)
    ivf.add(vectors)
    for nprobe in (1, 4, 16, 64):
        ivf.nprobe = nprobe
        _, found = ivf.search(probe_vectors, 10)
        recall = float(np.mean([len(set(found[i]) & set(exact_ids[i])) / 10
                                for i in range(n_queries)]))
        elapsed = timed(lambda: ivf.search(probe_vectors, 10))
        ann_rows.append((nlist, nprobe, recall, elapsed))
        print(f'{nlist:>7}{nprobe:>8}{recall:>13.4f}'
              f'{elapsed / n_queries * 1000:>11.4f}{flat_time / elapsed:>9.1f}×')

Подивись на останню колонку. У неї є рядки, де виграш менший за одиницю —
тобто наближений пошук **повільніший за точний перебір**, і повноту при цьому
все одно втрачає. Повнота й швидкість тут — не властивість бібліотеки, а дві
ручки одного налаштування, і в цієї сітки є явно погані кутки.

Числа часу в тебе будуть інші (машина, навантаження, версії). Відтворюється
**форма**: повнота росте з `nprobe` із загасанням, час росте майже лінійно, і
десь вони перетинаються.

## 11 · Збираємо RAG і рахуємо його стелю

RAG — це конструкція з чотирьох ланок: **пошук → добір → підказка → породження**.
Мовної моделі в цьому зошиті немає (і мережа тут заборонена), тому останню ланку
ми не виконуємо — але перші три складаються повністю, і саме в них лежить
головне обмеження.

Спершу зберемо підказку для конкретного запиту.

In [ ]:
def build_prompt(question, top_k=3, budget_chars=900):
    """Пошук -> добір у бюджет -> текст підказки."""
    scores = bm25_scores(tokens(question))
    order = np.argsort(-scores)[:top_k]
    chosen, used = [], 0
    for doc_id in order:                      # найкращий кандидат іде ПЕРШИМ
        piece = packages[doc_id]['description'].replace('\n', ' ')
        if used + len(piece) > budget_chars:
            piece = piece[:max(0, budget_chars - used)]
        if not piece:
            break
        chosen.append((doc_id, piece))
        used += len(piece)
    lines = ['Відповідай ЛИШЕ за наведеними фрагментами.',
             'Якщо відповіді в них немає — так і напиши.', '']
    for number, (doc_id, piece) in enumerate(chosen, 1):
        lines.append(f'[{number}] {packages[doc_id]["name"]}: {piece}')
    lines += ['', f'Питання: {question}']
    return '\n'.join(lines), [doc_id for doc_id, _ in chosen]

question = packages[probe]['summary']
prompt, used_ids = build_prompt(question)
print(prompt[:1100])
print('\n---')
print('потрібний документ у підбірці:', probe in used_ids)

Тепер головне число. Мовна модель не може відповісти по документу, якого їй
не дали. Отже, **повнота пошуку на глибині добору — це жорстка стеля всієї
конструкції**. Порахуємо її для кількох глибин.

In [ ]:
print(f'{"глибина добору":<18}{"стеля системи":>16}')
for depth in (1, 3, 5, 10, 20, 50, 100):
    ceiling = float((bm25_ranks <= depth).mean())
    print(f'{depth:<18}{ceiling:>16.4f}')
print()
print('Різниця між глибиною 5 і глибиною 100 — це весь запас, який може')
print('відіграти переранжування. Більше воно не дістане нізвідки.')

## 12 · Підсумок

Зберімо все, що зошит заміряв, в одну таблицю. Числа твої — на твоїй колекції.

In [ ]:
summary = [
    ('BM25', bm25_report['MRR'], bm25_report['recall@10']),
    ('TF-IDF косинус', tfidf_report['MRR'], tfidf_report['recall@10']),
    (f'щільний, стиснення {best_dim}',
     lsa_results[best_dim]['MRR'], lsa_results[best_dim]['recall@10']),
]
print(f'{"спосіб":<28}{"MRR":>9}{"recall@10":>12}')
for name, mrr, r10 in summary:
    print(f'{name:<28}{mrr:>9.4f}{r10:>12.4f}')
print()
print(f'на відкладеній колекції, документів: {M}')
print(f'  BM25             MRR {bm25_held["MRR"]:.4f}')
print(f'  навчений щільний MRR {dense_held["MRR"]:.4f}')
print()
print(f'весь зошит: {(time.process_time() - started_at)/60:.1f} хв процесорного часу')

### Що з цього варто винести

1. **BM25 — не «застаріла база», а незручний суперник.** У нього немає жодної
   навченої ваги, він працює з коробки, і побити його щільними векторами на
   малій своїй колекції важко.
2. **Стиснення матриці не додає знання.** Щільність сама собою не є семантикою:
   вектор знає рівно те, з чого його вчили.
3. **Змішувати варто різне.** Перш ніж робити гібрид, порахуй, скільки запитів
   слабша система розвʼязує, а сильніша ні. Якщо мало — гібрид не допоможе.
4. **Наближений пошук — це налаштування, а не режим «швидко».** Він уміє бути
   повільнішим за перебір.
5. **Стеля RAG стоїть на першій ланці.** Коли конструкція відповідає погано,
   спершу перевіряй, чи був потрібний документ у видачі.

---

## Завдання

**🟢 Рівень 1.** Додай до порівняння BM25 із іншими `k1` і `b` — перебери сітку
`k1 ∈ {0.6, 1.2, 2.0}` × `b ∈ {0, 0.5, 1.0}` і надрукуй MRR кожної клітинки.
**Зроблено, якщо** ти назвав словами, яка з двох ручок на твоїй колекції важить
більше й чому.

**🟡 Рівень 2.** Візьми запити, у яких правильний документ не потрапив у першу
десятку в жодного зі способів, і роздрукуй пʼять таких пар «запит → документ».
**Зроблено, якщо** ти сформулював гіпотезу, що в них спільного, і перевірив її
числом на всій колекції.

**🔴 Рівень 3.** Додай до навчання енкодера **важкі негативи**: замість
випадкових документів пачки підмішай ті, які BM25 поставив високо, але вони
неправильні. **Зроблено, якщо** ти показав таблицею, як змінились MRR і
recall@10 на відкладеній колекції, і чесно сказав, чи виграш узагалі є.